# Wie vervollständigt ein Sprachmodell einen Satz?

## Hintergrund

Große Sprachmodelle (LLMs, *Large Language Models*) wie ChatGPT, Gemma oder LLaMA wurden darauf trainiert, Text fortzusetzen. Im Kern lernen sie dabei: **Welches Wort folgt mit welcher Wahrscheinlichkeit auf einen gegebenen Text?**

In diesem Notebook befragen wir ein lokal laufendes Sprachmodell über **Ollama** – eine Software, die KI-Modelle direkt auf dem eigenen Rechner ausführt, ohne Internetverbindung.

---

## Was macht dieses Notebook?

Wir geben einen Satzanfang ein und fragen das Modell: *"Welche Wörter könnten als nächstes kommen?"*

**Wichtiger Hinweis zur Methode:** Das Modell gibt hier keine technisch exakten Token-Wahrscheinlichkeiten aus – es wird stattdessen gebeten, seine eigene Einschätzung in Worte zu fassen. Das Ergebnis ist also eine *Schätzung* des Modells über sich selbst, kein direkter Blick in seine internen Berechnungen.

---

**Voraussetzung:** Ollama muss installiert und gestartet sein (`ollama serve`), und das Modell `gemma3:1b` muss heruntergeladen sein (`ollama pull gemma3:1b`).

In [ ]:
# Die Bibliothek 'requests' wird benötigt, um HTTP-Anfragen an Ollama zu senden.
# Nur einmal ausführen – danach kann diese Zelle übersprungen werden.
pip install requests

In [ ]:
import requests

def ask_ollama_top_k(prompt, model="gemma3:1b", top_k=10):
    """
    Fragt ein lokal laufendes Ollama-Modell nach den wahrscheinlichsten
    Folgewörtern für einen gegebenen Satzanfang.

    Parameter:
        prompt  – der Satzanfang, den das Modell fortsetzen soll
        model   – Name des Ollama-Modells (muss lokal verfügbar sein)
        top_k   – Anzahl der gewünschten Vorschläge
    """

    # Ollama läuft lokal auf Port 11434. /api/generate ist der Endpunkt für Textgenerierung.
    ollama_url = "http://localhost:11434/api/generate"

    # Wir formulieren eine Aufgabe in natürlicher Sprache für das Modell (sog. Prompt Engineering).
    # Das Modell wird gebeten, seine eigene Einschätzung zu nennen – keine echten Logits!
    full_prompt = (
        f"Was sind die {top_k} wahrscheinlichsten nächsten Wörter "
        f"nach folgendem Text:\n\"{prompt}\"\n"
        f"Antworte nur mit einer nummerierten Liste von Wörtern ohne weitere Erklärungen."
    )

    try:
        # HTTP POST-Anfrage an Ollama senden
        # stream=False bedeutet: wir warten auf die vollständige Antwort (kein Streaming)
        response = requests.post(ollama_url, json={
            "model": model,
            "prompt": full_prompt,
            "stream": False
        })

        if response.status_code == 200:
            # Aus der JSON-Antwort das Textfeld "response" extrahieren
            return response.json().get("response", "").strip()
        else:
            return f"Server-Fehler: {response.status_code} – läuft Ollama? (ollama serve)"

    except requests.exceptions.ConnectionError:
        return "Verbindungsfehler: Ollama scheint nicht zu laufen. Bitte 'ollama serve' starten."
    except Exception as e:
        return f"Unbekannter Fehler: {str(e)}"


## Ausprobieren – Satzanfang eingeben

Gib einen beliebigen Satzanfang ein (z. B. *"Die Sonne"*, *"Künstliche Intelligenz ist"*, *"Heute war ein"*) und schaue, welche Folgewörter das Modell vorschlägt.

In [ ]:
# Eingabe: Satzanfang
prompt = input("Bitte Satzanfang eingeben: ")

In [ ]:
# Anfrage an das Modell senden und Ergebnis ausgeben
result = ask_ollama_top_k(prompt)

print(f"Satzanfang: \"{prompt}\"")
print(f"Modell: gemma3:1b\n")
print("Geschätzte Folgewörter laut Modell:")
print("-" * 35)
print(result)

---

## Aufgaben und Experimente

### Aufgabe 1 – Beobachtung
Teste mindestens **drei verschiedene Satzanfänge**. Notiere die Ergebnisse.
- Welche Muster erkennst du?
- Wirkt die Liste wie eine sinnvolle Fortsetzung?

### Aufgabe 2 – Parameter verändern
In der nächsten Zelle kannst du `top_k` und das Modell anpassen:
- Was ändert sich, wenn du `top_k=3` oder `top_k=20` verwendest?
- Probiere ein anderes Modell (z. B. `"llama3.2:1b"`, falls installiert). Unterscheiden sich die Vorschläge?

### Aufgabe 3 – Kritische Reflexion
Beantworte folgende Fragen schriftlich oder im Unterrichtsgespräch:
1. Glaubst du, das Modell *versteht* den Satz – oder erkennt es nur statistische Muster?
2. Was bedeutet es, dass das Modell seine eigenen Wahrscheinlichkeiten nur *schätzen* kann, sie aber nicht direkt kennt?
3. Welche Probleme könnten entstehen, wenn ein Sprachmodell immer das *wahrscheinlichste* Wort wählt?

In [ ]:
# Experiment: top_k und Modell anpassen
# Verändere die Werte und vergleiche die Ergebnisse!

eigener_prompt = input("Satzanfang: ")
ergebnis = ask_ollama_top_k(eigener_prompt, model="gemma3:1b", top_k=10)

print(f"\nErgebnis für: \"{eigener_prompt}\"")
print("-" * 35)
print(ergebnis)

---

## Weiterführende Überlegung

Echte Sprachmodelle wählen das nächste Wort nicht immer deterministisch (also immer das Wahrscheinlichste), sondern **zufallsbasiert gewichtet** – sonst wären alle generierten Texte identisch. Dieses Verfahren nennt sich **Sampling**.

Parameter wie **Top-K-Sampling** oder **Temperature** steuern dabei, wie "kreativ" oder "vorhersehbar" das Modell antwortet:

| Parameter | Wirkung |
|---|---|
| **Temperature → 0** | Modell wählt fast immer das wahrscheinlichste Wort → vorhersehbar |
| **Temperature → 1** | Modell wählt gleichmäßiger aus vielen Wörtern → kreativer, aber weniger kohärent |
| **Top-K klein** | Nur die K wahrscheinlichsten Wörter kommen überhaupt in Frage |
| **Top-K groß** | Auch seltenere Wörter sind möglich → mehr Variation |

Diese Parameter kannst du auch an Ollama übergeben – das ist ein möglicher nächster Schritt!

In [ ]:
# Freier Bereich für eigene Experimente